In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q03/annotated/checkpoints/pre_cell_3.pickle

In [ ]:
%%RecordEvent
%%time
### cell 3 ###

### cell 3 optimized ###

# use a literal date string so cudf can do the comparison on the GPU
date = "1995-03-04"

res = (
    customer
      .loc[customer.C_MKTSEGMENT == "HOUSEHOLD", ["C_CUSTKEY"]]
    .merge(
        orders
          .loc[orders.O_ORDERDATE < date,
               ["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_SHIPPRIORITY"]],
        on="C_CUSTKEY",
    )
    .merge(
        lineitem
          .loc[lineitem.L_SHIPDATE > date,
               ["L_ORDERKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]],
        left_on="O_ORDERKEY",
        right_on="L_ORDERKEY",
    )
    .assign(TMP=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT))
    .groupby(
        ["L_ORDERKEY", "O_ORDERDATE", "O_SHIPPRIORITY"],
        as_index=False,
        sort=False,
    )
    ["TMP"].sum()
    .sort_values("TMP", ascending=False)
    .loc[:, ["L_ORDERKEY", "TMP", "O_ORDERDATE", "O_SHIPPRIORITY"]]
)


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q03/rewritten/o4_mini_high/checkpoints/post_cell_3_try_4.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/tpch/notebooks/q03/opt_cell_exec_info_3_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[3], f)


In [ ]:
opt_output = Out.get(4)